# 査読コメント対応 — 実現距離 R(x*) と Z(x*) のトレードオフ

Jinwoo 先生のコメントのうち、**横軸を課した上限 D ではなく、各最適解が実際に到達した
区域内最大距離 R(x\*) に取り替えて Z(x\*) との関係を見せる**という部分に答えるための
ノートブック。依頼された5項目を上から順に確認していく。

| | 問い | 答えを出すセル |
|---|---|---|
| 1 | 36ケースの割当 x\* は保存されているか。無ければ計算する | 第1章 |
| 2 | 区域ごとの R_k(x\*) と その最大 R(x\*)。全ケースで R(x\*) ≤ D か | 第2章 |
| 3 | 36ケース＋現行区割り1行＝**37行**の表（D, M, Z, R, 削減率） | 第3章 |
| 4 | 横軸 R(x\*)、縦軸 Z(x\*)、M 別マーカー、パレートフロンティアの図 | 第4章 |
| 5 | 1ケースあたりの求解時間（L 掃引をやるかの判断材料） | 第5章 |

## 結論の先出し

- **再最適化は不要**。36ケースすべての割当行列が保存済みなので、R(x\*) は距離行列を
  引くだけで求まる。Gurobi はこのノートブックでは使わない。
- R(x\*) はほぼ全ケースで D の直下に張り付く（制約が効いている）。したがって
  「D を振って得たフロンティア」は実質 R ≈ D の対角線になる。これは Eq. (18) が
  **最適値を与える割当の中で R を最小化していない**ことの裏返しで、
  Jinwoo 先生自身が本文中コメントで指摘している点と同じ。
- 副産物として、**D = 15 km の最適解は現行の行政区割りを両方の軸で下回る**
  （R: 16.33 → 14.95 km、Z: 2.588 → 2.087）。区域を広げずに契約を減らせる、
  という主張が1点で言える。

## 使い方

- **1セルずつ上から順に実行する**（「すべて実行」でも同じ）。セルは上から下へ
  一方向にしか依存しない。
- 計算そのものは `scripts/` 配下のプログラムが行い、このノートブックはそれを
  呼び出して結果を確かめるだけ。計算式はノートブック側に書かれていない。
- 各確認は `check(...)` が ✅ / ❌ を出し、最後の章で一覧表にまとめる。
- 出力先は既定では作業用フォルダ `outputs/reviewer_pareto/<実行時刻>/`。
  論文が参照する `figures/`・`outputs/` を更新したいときだけ、
  第0章の `UPDATE_CANONICAL` を `True` にする。

## 0. セットアップ

以降のすべてのセルが使う土台を用意する。`notebooks/pipeline_walkthrough.ipynb` と
同じ道具立て（`run` / `check` / `show`）をそのまま使う。

| 関数 | 役割 |
|---|---|
| `run(スクリプト名, 引数...)` | `scripts/` のプログラムを1本実行し、出力をそのまま表示する。失敗したら止まる |
| `check(章, 確認名, 真偽, 補足)` | 確認結果を1件記録して ✅ / ❌ を表示する |
| `show(画像パス)` | 生成された図をノートブック内に表示する |

In [ ]:
from __future__ import annotations

import datetime
import os
import pickle
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image, display

# --- 1. リポジトリの位置を割り出す ------------------------------------------
# notebooks/ から開いてもリポジトリ直下から開いても同じように動くよう、
# scripts/ フォルダが見えるところまで1階層さかのぼる。
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "scripts").is_dir():
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / "scripts").is_dir(), f"リポジトリルートが見つからない: {Path.cwd()}"
sys.path.insert(0, str(REPO_ROOT / "src"))

# --- 2. この実行専用の出力フォルダを作る ------------------------------------
PY = sys.executable
RUN_ID = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
OUT = REPO_ROOT / "outputs" / "reviewer_pareto" / RUN_ID
OUT.mkdir(parents=True, exist_ok=True)

# --- 3. 図表の出力先を決める ------------------------------------------------
# 既定では作業用フォルダに出力し、論文が参照するファイルには触れない。
UPDATE_CANONICAL = False
FIG_DIR = (REPO_ROOT / "figures") if UPDATE_CANONICAL else OUT
TAB_DIR = (REPO_ROOT / "outputs") if UPDATE_CANONICAL else OUT
FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)

# --- 4. 以降のセルが使う共通の道具 ------------------------------------------
CHECKS: list[dict] = []


def check(stage: str, name: str, passed: bool, detail: str = "") -> bool:
    """確認結果を1件記録して ✅ / ❌ を表示する。"""
    CHECKS.append({"stage": stage, "check": name, "result": "PASS" if passed else "FAIL",
                   "detail": detail})
    print(f"{'✅' if passed else '❌'} [{stage}] {name}" + (f" — {detail}" if detail else ""))
    return passed


def run(script: str, *args, timeout: int = 1800) -> str:
    """scripts/ 配下のプログラムを1本実行し、その出力をそのまま表示する。"""
    cmd = [PY, str(REPO_ROOT / "scripts" / script), *map(str, args)]
    env = os.environ.copy()
    src = str(REPO_ROOT / "src")
    env["PYTHONPATH"] = src + (os.pathsep + env["PYTHONPATH"] if env.get("PYTHONPATH") else "")
    print(f"$ python scripts/{script} " + " ".join(map(str, args)))
    r = subprocess.run(cmd, cwd=REPO_ROOT, capture_output=True, text=True, env=env, timeout=timeout)
    print(r.stdout)
    if r.returncode != 0:
        print(r.stderr, file=sys.stderr)
        raise RuntimeError(f"{script} が終了コード {r.returncode} で失敗")
    return r.stdout


def show(path, width: int = 820) -> None:
    """生成された図をノートブック内に表示する。"""
    p = Path(path)
    display(Image(filename=str(p), width=width)) if p.exists() else print(f"(未生成: {p})")


# このノートブックが使う入力ファイル（すべてリポジトリに入っている）
SOLUTIONS_PKL = REPO_ROOT / "data/processed/districting_solutions_all36.pkl"
DISTANCE_PKL = REPO_ROOT / "data/processed/distance_matrix_322_20251208.pkl"
BRIDGES_CSV = REPO_ROOT / "data/processed/target_rc_bridges_322.csv"
RESULTS_CSV = REPO_ROOT / "data/processed/optimization_results_exact_objective.csv"

print("REPO_ROOT :", REPO_ROOT)
print("作業用出力:", OUT.relative_to(REPO_ROOT))
print("図の出力先:", FIG_DIR.relative_to(REPO_ROOT), "/ 表の出力先:", TAB_DIR.relative_to(REPO_ROOT))
print("UPDATE_CANONICAL =", UPDATE_CANONICAL, "（True なら figures/・outputs/ の正本を更新する）")

## 1. 【問1】36ケースの割当 x\* は保存されているか

**答え: 保存されている。再計算は不要。**

地域分割最適化を解く `scripts/run_gurobi_districting.py` は、結果CSVとは別に
`--solutions-output` で**割当行列そのもの**を pickle に書き出す。実行可能だった
36ケース分が `data/processed/districting_solutions_all36.pkl` に入っており、
これはリポジトリに含まれている（Git 管理下）。

pickle の中身は `(D, M) → {"assignment", "row", "order"}` の辞書で、

- `assignment` — 322×M の 0/1 行列。行が橋梁、列が管理エリア
- `order` — 行の並び順に対応する `shisetsu_id` のリスト。距離行列の `order` と同じ並び
- `row` — そのケースの結果CSV1行分（目的値・地域別橋梁数・求解時間など）

このセルでは、36ケース揃っていること、行列の形が正しいこと、
各橋梁がちょうど1エリアに割り当てられていること、そして
**正本の結果CSVと同じ (D, M) の集合であること**を確認する。

In [ ]:
with SOLUTIONS_PKL.open("rb") as f:
    solutions = pickle.load(f)

results = pd.read_csv(RESULTS_CSV)
solved = results[results["Status"] == 2]

print(f"{SOLUTIONS_PKL.relative_to(REPO_ROOT)}")
print(f"  ケース数: {len(solutions)}")
print(f"  (D, M)  : {sorted(solutions.keys())}\n")

check("1.割当", "36ケース揃っている", len(solutions) == 36, f"{len(solutions)} ケース")

# 正本CSVの (D, M) 集合と一致するか。片方だけにあるケースがあれば図表がずれる。
keys_pkl = {(float(d), int(m)) for d, m in solutions.keys()}
keys_csv = {(float(r["MaxDistance"]), int(r["M"])) for _, r in solved.iterrows()}
check("1.割当", "正本CSVと同じ (D, M) の集合",
      keys_pkl == keys_csv,
      f"pklのみ={sorted(keys_pkl - keys_csv)} / CSVのみ={sorted(keys_csv - keys_pkl)}")

# 割当行列の形と、各橋梁がちょうど1エリアに入っていること
shape_ok, onehot_ok, order_ok = True, True, True
for (d, m), entry in solutions.items():
    a = np.asarray(entry["assignment"])
    shape_ok &= a.shape == (322, m)
    onehot_ok &= bool((a.sum(axis=1) == 1).all())
    order_ok &= len(entry["order"]) == 322

check("1.割当", "行列の形が 322×M", shape_ok)
check("1.割当", "各橋梁がちょうど1エリアに属する", onehot_ok)
check("1.割当", "order が 322 件", order_ok)

# 保存済みの結果行が、割当から再計算した地域別橋梁数と合うか（ここが崩れていたら
# pkl と CSV が別の実行のものということになる）
mismatch = []
for (d, m), entry in solutions.items():
    a = np.asarray(entry["assignment"])
    recomputed = sorted((int(a[:, k].sum()) for k in range(m)), reverse=True)
    stored = sorted((int(v) for v in str(entry["row"]["RegionCounts"]).split(";") if v), reverse=True)
    if recomputed != stored:
        mismatch.append((d, m))
_ = check("1.割当", "地域別橋梁数が保存行と一致", not mismatch, f"不一致 ={mismatch}")

## 2. 【問2】R_k(x\*) と R(x\*) を計算し、R(x\*) ≤ D を確認する

管理エリア k の**直径**（区域内で最も離れた2橋の距離）と、その全エリア最大を

$$R_k(x) = \max\{\, d_{ij} : i, j \in k \,\}, \qquad R(x) = \max_k R_k(x)$$

と定義する。橋梁が1つだけのエリアは $R_k = 0$ とする。

### 2-1. 距離が MIP で使ったものと同じ大円距離か確認する

R(x\*) は「最適化が実際に満たした制約の値」でなければ意味がないので、
**最適化に渡したのと同一の距離行列**を使う必要がある。
`data/processed/distance_matrix_322_20251208.pkl` がその行列で、
`run_gurobi_districting.py` が `--distance-matrix` として読むのもこれである。

中身が大円距離であることは、橋梁CSVの緯度経度から haversine 距離
（地球半径 6371.0088 km、`src/bundling_analysis/distance_cache.py`）を
計算し直して突き合わせれば確かめられる。

In [ ]:
from bundling_analysis.distance_cache import haversine_km

with DISTANCE_PKL.open("rb") as f:
    dm = pickle.load(f)
order = [str(x) for x in dm["order"]]
D_MATRIX = np.asarray(dm["d_core"], dtype=float)

print(f"{DISTANCE_PKL.relative_to(REPO_ROOT)}")
print(f"  形: {D_MATRIX.shape}")
print(f"  最大距離（322橋全体の直径）: {D_MATRIX.max():.3f} km\n")

# 橋梁CSVの緯度経度から大円距離を計算し直して突き合わせる
bridges = pd.read_csv(BRIDGES_CSV)
b = bridges.assign(_sid=bridges["shisetsu_id"].astype(str)).set_index("_sid").loc[order]
lat = b["緯度"].to_numpy(float)
lon = b["経度"].to_numpy(float)
recomputed = haversine_km(lat[:, None], lon[:, None], lat[None, :], lon[None, :])

max_gap = float(np.abs(recomputed - D_MATRIX).max())
print(f"  再計算した haversine 距離との最大差: {max_gap:.3e} km")

check("2.距離", "距離行列が正方・対称", D_MATRIX.shape[0] == D_MATRIX.shape[1]
      and np.allclose(D_MATRIX, D_MATRIX.T))
check("2.距離", "対角が 0", bool(np.allclose(np.diag(D_MATRIX), 0.0)))
check("2.距離", "緯度経度からの大円距離と一致（< 1e-6 km）", max_gap < 1e-6,
      f"最大差 {max_gap:.3e} km")
_ = check("2.距離", "橋梁CSVと距離行列の shisetsu_id が同一集合",
          sorted(bridges["shisetsu_id"].astype(str)) == sorted(order))

### 2-2. R_k(x\*)・R(x\*) を計算し、距離制約が守られているか確認する

計算は `scripts/compute_realized_radius.py` が行う。このプログラムは

1. 保存済み割当と距離行列から各エリアの直径 $R_k$ と全体の $R(x^*)$ を出す
2. 割当から地域別橋梁数と厳密目的値を計算し直し、**保存行と一致するか検証**する
3. **全ケースで $R(x^*) \le D$ か検証**する
4. 現行の行政区割り（橋梁を管理者でまとめたもの）を1行足して表にする

の順に動く。2 と 3 のどちらかが崩れたらエラーで止まる（`--no-verify` で無視可）。

In [ ]:
RADIUS_CSV = TAB_DIR / "realized_radius.csv"

out = run("compute_realized_radius.py",
          "--solutions", SOLUTIONS_PKL,
          "--distance-matrix", DISTANCE_PKL,
          "--bridges", BRIDGES_CSV,
          "--bundle-limit", 5,
          "--output", RADIUS_CSV)

radius = pd.read_csv(RADIUS_CSV)
opt = radius[radius["Districting"] == "optimized"]

check("2.距離", "全36ケースで R(x*) <= D",
      bool((opt["RealizedRadius"] <= opt["MaxDistance"] + 1e-6).all()),
      f"R-D の最大は {float((opt['RealizedRadius'] - opt['MaxDistance']).max()):.6f} km（負なら制約内）")
check("2.距離", "検証がエラーなく完了（件数・目的値が保存行と一致）", "検証失敗" not in out)

# D に対して R がどれだけ余っているか。ほぼ 0 なら距離制約が効いている。
print("\nD ごとの余裕 D - R(x*) [km]:")
display(opt.groupby("MaxDistance")["Slack_D_minus_R"].agg(["min", "max"]).round(3))
print("→ D>=40 では 322 橋全体の直径 39.59 km に達し、制約が効かなくなる（余裕が開く）。")

## 3. 【問3】37行の表

行は **実行可能だった (D, M) の36ケース＋現行区割り1行**。
列は依頼どおり D・M・Z(x\*)・R(x\*)・現行区割り比の削減率で、
加えて求解時間と、非劣（パレート最適）かどうかの印を持たせてある。

現行区割りの行は、322橋をそれぞれの**管理者（市町）**でまとめたもので、
距離上限を課していないので D 欄は空にしてある。この行の Z が削減率の分母になる。

| 列 | 意味 |
|---|---|
| `Districting` | `optimized`（最適化した区割り）/ `current`（現行の行政区割り） |
| `MaxDistance` | 課した距離上限 D [km]。現行区割りには無い |
| `M` | 管理エリア数 |
| `ObjectiveValue_Exact` | Z(x\*) = 期待年間契約件数（閉形式による厳密値） |
| `RealizedRadius` | R(x\*) = 実現した区域内最大距離 [km] |
| `Slack_D_minus_R` | D − R(x\*)。0 に近いほど距離制約が効いている |
| `Reduction(%)` | 現行区割り比の削減率 |
| `ElapsedSeconds` | そのケースの求解時間（第5章で使う） |
| `Nondominated` | (R, Z) 両方を小さくする意味で非劣なら 1 |
| `RegionDiameters` | エリア別の $R_k$（大きい順） |
| `RegionCounts` | エリア別の橋梁数（大きい順） |

In [ ]:
check("3.表", "37行（36ケース＋現行区割り1行）", len(radius) == 37, f"{len(radius)} 行")

# 依頼された5列を主役にした表示用の表
view = radius[["Districting", "MaxDistance", "M", "ObjectiveValue_Exact",
               "RealizedRadius", "Reduction(%)", "ElapsedSeconds", "Nondominated"]].copy()
view.columns = ["区割り", "D [km]", "M", "Z(x*)", "R(x*) [km]", "削減率 [%]", "求解 [s]", "非劣"]
view["Z(x*)"] = view["Z(x*)"].round(4)
view["R(x*) [km]"] = view["R(x*) [km]"].round(2)
view["削減率 [%]"] = view["削減率 [%]"].round(1)

display(view.style.hide(axis="index").format(na_rep="—"))
print(f"\n表の実体: {RADIUS_CSV}")

### 3-2. 非劣解だけを取り出す

D を振って得られる (R, Z) の非劣集合。これが「D を振るだけで得られるフロンティア」で、
Eq. (17) の単位依存の重み α を持ち込まずに描ける、というのが Jinwoo 先生の趣旨。

現行区割りとの比較も、ここに並べれば一目で分かる。

In [ ]:
frontier = radius[radius["Nondominated"] == 1].sort_values("RealizedRadius")
current = radius[radius["Districting"] == "current"].iloc[0]

print(f"現行の行政区割り: R = {current['RealizedRadius']:.2f} km, "
      f"Z = {current['ObjectiveValue_Exact']:.4f}（M = {int(current['M'])} 市町）\n")

front_view = frontier[["MaxDistance", "M", "ObjectiveValue_Exact", "RealizedRadius",
                       "Reduction(%)"]].copy()
front_view.columns = ["D [km]", "M", "Z(x*)", "R(x*) [km]", "削減率 [%]"]
display(front_view.round({"Z(x*)": 4, "R(x*) [km]": 2, "削減率 [%]": 1})
        .style.hide(axis="index"))

# 現行区割りを (R, Z) 両方で下回る解があるか。あれば「区域を広げずに契約を減らせる」と言える。
better = frontier[(frontier["RealizedRadius"] <= current["RealizedRadius"]) &
                  (frontier["ObjectiveValue_Exact"] <= current["ObjectiveValue_Exact"])]
_ = check("3.表", "現行区割りを (R, Z) 両方で下回る最適解が存在する", len(better) > 0,
          f"{len(better)} 件 — " + ", ".join(f"D={r['MaxDistance']:.0f}/M={int(r['M'])}"
                                             for _, r in better.iterrows()))

## 4. 【問4】横軸 R(x\*)・縦軸 Z(x\*) の図

`scripts/plot_pareto_radius.py` が第3章の表から図を描く。

- マーカーの色は**管理エリア数 M 別**。色の対応は `figures/dm_sensitivity`（現 FIG. 6）と
  共通の定義（`src/bundling_analysis/plotting_utils.py` の `M_COLORS`）を使うので、
  2つの図を並べても同じ M が同じ色になる
- 灰色の階段線が**パレートフロンティア**。点と点を直線で結ばないのは、
  フロンティアが点の集合であって連続曲線ではないため。階段は
  「区域内最大距離を R まで許したときに達成できる最小の Z」を表す
- 黒い星が**現行の行政区割り**
- `--annotate-d` を付けると、各フロンティア点にそれを生んだ D を添える

出力は PNG・SVG・PDF の3種類。本文用はベクタ（PDF）を使う。

**FIG. 6 と差し替えるか併置するかは水谷先生の判断待ち**なので、
既定では `figures/` を上書きせず作業用フォルダにだけ出す。
差し替えを決めたら第0章の `UPDATE_CANONICAL = True` で正本を更新できる。

In [ ]:
FIG_STEM = FIG_DIR / "pareto_radius"

run("plot_pareto_radius.py",
    "--input", RADIUS_CSV,
    "--annotate-d",
    "--output-stem", FIG_STEM)

show(FIG_STEM.with_suffix(".png"))

for ext in ("png", "svg", "pdf"):
    p = FIG_STEM.with_suffix(f".{ext}")
    check("4.図", f"{ext.upper()} が生成された", p.exists(), str(p.relative_to(REPO_ROOT)))

### 4-2. 参考: 現 FIG. 6（横軸 D）との対比

同じ36ケースを、横軸だけ D に戻したもの。**点の位置がほとんど変わらない**ことが、
「R(x\*) はほぼ D に張り付いている」ことの視覚的な確認になる。

裏を返すと、Eq. (18) は最適値を与える割当の中で R を最小化していないので、
この図の R はソルバが返した実現値であって、その Z を達成する**最小の**距離ではない。
本当に R が最小の非劣解が欲しければ、Z\* を固定して R を最小化する2段階
（辞書式）最適化が要り、そこは Gurobi での解き直しになる。

In [ ]:
run("plot_dm_sensitivity.py",
    "--input", RESULTS_CSV,
    "--bridges", BRIDGES_CSV,
    "--bundle-limit", 5,
    "--output-stem", OUT / "dm_sensitivity_reference")

show(OUT / "dm_sensitivity_reference.png")

## 5. 【問5】1ケースあたりの求解時間

L を変えた掃引（Jinwoo 先生の提案の後半、「M を固定して L を変え、
バンドリング能力がフロンティアをどう押し下げるか見る」）をやるかどうかの判断材料。

求解時間は結果CSVの `ElapsedSeconds` に入っている。全整数PWL・
`--warm-start-min-m 4 --mip-gap 0.005` での実測値である。

**先に結論**: 費用はほぼ全部 M に乗っている。M ≤ 3 なら1ケース数秒で終わり、
M = 6・D = 15 の1ケースだけが2時間かかる。Jinwoo 先生の言う「M を固定して」
という枠組みは、固定する M 次第で所要時間が3桁変わる。

なお**実行可能な (D, M) の組は L に依存しない**（距離制約と「各エリアに1橋以上」
だけで決まる）ので、L を変えても同じ36ケースがそのまま使える。

In [ ]:
times = results[results["Status"] == 2][["MaxDistance", "M", "ElapsedSeconds"]].copy()

print(f"全{len(times)}ケース合計: {times['ElapsedSeconds'].sum() / 3600:.2f} 時間")
print(f"中央値: {times['ElapsedSeconds'].median():.1f} 秒 / "
      f"最大: {times['ElapsedSeconds'].max():.0f} 秒\n")

by_m = times.groupby("M")["ElapsedSeconds"].agg(["count", "median", "max", "sum"])
by_m.columns = ["ケース数", "中央値 [s]", "最大 [s]", "合計 [s]"]
display(by_m.round(1))

print("\n所要時間の上位5ケース:")
top = times.nlargest(5, "ElapsedSeconds").copy()
top.columns = ["D [km]", "M", "求解 [s]"]
display(top.style.hide(axis="index").format({"求解 [s]": "{:.1f}"}))

# L 掃引の見積もり。実行可能な (D, M) は L に依らないので、同じケース集合を
# L 水準ごとに解き直すことになる。M を固定すればその M の行だけが効く。
print("\nL を1水準足したときの見積もり（同じケース集合を解き直す前提）:")
for m in sorted(times["M"].unique()):
    sub = times[times["M"] == m]
    d_range = f"D={sub['MaxDistance'].min():.0f}–{sub['MaxDistance'].max():.0f}"
    print(f"  M={m} 固定 ({len(sub)}ケース, {d_range}): {sub['ElapsedSeconds'].sum():>8.0f} 秒"
          f"  = {sub['ElapsedSeconds'].sum() / 3600:>5.2f} 時間")
print(f"  全36ケース             : {times['ElapsedSeconds'].sum():>8.0f} 秒"
      f"  = {times['ElapsedSeconds'].sum() / 3600:>5.2f} 時間")

### 5-2. L 掃引をやる場合のコマンド

`scripts/run_gurobi_districting.py` は `--bundle-limit` を既に持っているので、
**コードの改修は不要**。L 水準ごとに1回ずつ呼べばよい。
本文主結果の L = 5 は既存の36ケースをそのまま流用できるので、
新しく解くのは他の水準だけで済む。

Gurobi はこのマシンには入っていないので、ライセンスのある別PCで実行する
（`docs/remote_gurobi_setup.md`、`docs/multi_pc_git_python_notes.md`）。
長時間ジョブはセッション切断で2回落ちた実績があるため、
デタッチして走らせること（Windows ならタスクスケジューラ、Linux なら nohup / tmux）。

```bash
# 例: M=3 固定・D=25〜50（数秒で終わる）。L=2,3,10 を順に解く
for L in 2 3 10; do
  python scripts/run_gurobi_districting.py \
      --pwl all --bundle-limit $L \
      --cases 25:3 30:3 35:3 40:3 45:3 50:3 \
      --threads 8 --mip-gap 0.005 \
      --output outputs/lsweep_M3_L${L}.csv \
      --solutions-output outputs/lsweep_M3_L${L}_solutions.pkl
done
```

M = 6 に固定すれば D = 15〜50 の全域を1本の曲線にできる（対象6市町と同数という
解釈も付く）が、上の表のとおり1水準あたり約2.2時間かかる。

解き終わったら、その `--solutions-output` をこのノートブックの第2章に
そのまま渡せば、L 別の R(x\*) 表と図が同じ手順で作れる。

```python
run("compute_realized_radius.py",
    "--solutions", REPO_ROOT / "outputs/lsweep_M3_L3_solutions.pkl",
    "--bundle-limit", 3,                      # 削減率の分母も L=3 で計算し直される
    "--output", TAB_DIR / "realized_radius_L3.csv")
```

## 6. 確認の一覧

このノートブックで行った確認をまとめる。**すべて PASS であることが、
第3章の表と第4章の図を論文・返信に使える条件**である。

In [ ]:
summary = pd.DataFrame(CHECKS)
display(summary.style.hide(axis="index"))

n_fail = int((summary["result"] == "FAIL").sum())
print(f"\n{len(summary) - n_fail}/{len(summary)} PASS")
if n_fail:
    print("❌ FAIL があります。上の該当セルの出力を確認してください。")
else:
    print("✅ すべて PASS。")
    print(f"\n成果物:")
    print(f"  表 (37行): {RADIUS_CSV}")
    print(f"  図        : {FIG_STEM}.png / .svg / .pdf")